In [60]:
# Imports
import os
import sys
from IPython.display import display

from py_scripts.parser_registry import get_parse_files, list_parsers

# Backend selection ('music21' | 'partitura')
PARSING_BACKEND = 'music21'

# Select parser function lazily by backend name
parse_files = get_parse_files(PARSING_BACKEND)

# Configuration
FILTER_ZERO_DURATION = True # filter out notes with duration 0 (grace notes)
ADJUST_FRACTIONAL_DURATION = True # rounds Duration, Local Onset, and Global Onset to 3 decimals
PARSE_ENHARMONIC = True     # add 'Pitch Enharmonic' column grounded in source notation
PLOTTING_BACKEND = 'bokeh'  # 'plt' | 'bokeh' | 'none'
SHOW_MEASURE_LINES = True   # draw red vertical lines at measure starts

DISPLAY_PREVIEW = True      # show a preview of the parsed data as df
PREVIEW_ROWS = 20          # number of rows to show in the preview

CLEANUP_REMOTE = True       # if source is a URL, delete temp file after parsing
RETURN_PLOTS = False        # include plot objects in results under 'plot'
STRIP_TIES = True           # merge tied notes into single notes (music21 backend only)
ALIGN_ACCIDENT_SCHEMA = True # align accidentals on both parsing backends to the same schema (Pitch Enharmonic is parsed only in b/# format)

# List of file sources (URLs or local paths)
FILE_SOURCES = [
    # 'https://analyse.hfm-weimar.de/database/02/PrJode_Jos0302_COM_1-5_MissaDapac_002_00006.xml',
    # 'https://raw.githubusercontent.com/humdrum-tools/bach-wtc-fugues/refs/heads/master/kern/wtc1f04.krn',
    # 'https://raw.githubusercontent.com/humdrum-tools/bach-wtc/refs/heads/main/kern/wtc1f22.krn',
    'https://raw.githubusercontent.com/piasteuck/winterreise-analysis/refs/heads/main/Schubert_Winterreise_Dataset_v2-0/01_RawData/score_musicxml/Schubert_D911-20.xml'
    # 'https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.0/Music/Complete_examples/Bach-JS_Ein_feste_Burg.mei',
    # 'https://raw.githubusercontent.com/trompamusic-encodings/Beethoven_Op31_No3_HenleUrtext/refs/heads/master/Beethoven_Op31_No3_3-HenleUrtext.mei',
    # 'https://raw.githubusercontent.com/humdrum-tools/bach-wtc/refs/heads/main/kern/wtc2f05.krn',
    # 'C:/Users/egorp/OneDrive/Desktop/weimar_ftp_backup/dokuwiki/database/05/CaSe_11_UNSP_UNSP_Danksagenw_005_00011.xml'  # If you prefer backslashes, make it a raw string: r'C:\Users\egorp\...'
]

# Plot sizing
PLOT_SIZE_X = 900  # width in pixels
PLOT_SIZE_Y = 600   # height in pixels

# Zoom tool dimensions: 'both' | 'width' (x-only) | 'height' (y-only)
# Bokeh only!
ZOOM_DRAG_DIM = 'both'
ZOOM_WHEEL_DIM = 'width'

# test voice colors
COLORIZE_VOICES = True 

# Parse and visualize
results, dfs_by_name, df_processed = parse_files(
    FILE_SOURCES,
    filter_zero_duration=FILTER_ZERO_DURATION,
    adjust_fractional_duration=ADJUST_FRACTIONAL_DURATION,
    parse_enharmonic=PARSE_ENHARMONIC,
    backend=PLOTTING_BACKEND,
    show_measure_lines=SHOW_MEASURE_LINES,
    display_preview=DISPLAY_PREVIEW,
    preview_rows=PREVIEW_ROWS,
    cleanup_remote=CLEANUP_REMOTE,
    return_plots=RETURN_PLOTS,
    plot_width=PLOT_SIZE_X,
    plot_height=PLOT_SIZE_Y,
    zoom_drag_dim=ZOOM_DRAG_DIM,
    zoom_wheel_dim=ZOOM_WHEEL_DIM,
    strip_ties=STRIP_TIES,
    align_accident_schema=ALIGN_ACCIDENT_SCHEMA,
    colorize_voices=COLORIZE_VOICES
)

# Show quick summary
print('Parsed DataFrames:')
for item in results:
    name = item['name']
    df = item['df']
    pos_dur = df.loc[df['Duration'] > 0, 'Duration'] if 'Duration' in df.columns else None
    min_dur_str = str(float(pos_dur.min())) if pos_dur is not None and len(pos_dur) > 0 else 'n/a'
    print(f"- {name}: rows={len(df)}, unique_pitches={df['MIDI'].nunique()}, min_duration={min_dur_str}")


Processing (music21): Schubert_D911-20.xml -> 00_schubert_d911_20


,Measure,Local Onset,Global Onset,Duration,Pitch,Pitch Enharmonic,MIDI,Voice
0,0,0.00,0.00,0.25,G4,G4,67,Piano / Piano (2)
1,0,0.00,0.00,0.25,A#3,Bb3,58,Piano / Piano (2) / Voice 4
2,0,0.25,0.25,0.25,A4,A4,69,Piano / Piano (2)
3,0,0.25,0.25,0.25,C4,C4,60,Piano / Piano (2) / Voice 4
4,1,0.00,0.50,0.50,G4,G4,67,Piano / Piano (2)
5,1,0.00,0.50,0.50,A#4,Bb4,70,Piano / Piano (2)
6,1,0.00,0.50,0.50,G3,G3,55,Piano / Piano (2) / Voice 5
7,1,0.00,0.50,0.50,D4,D4,62,Piano / Piano (2) / Voice 5
8,1,0.50,1.00,0.50,G4,G4,67,Piano / Piano (2)
9,1,0.50,1.00,0.50,A#4,Bb4,70,Piano / Piano (2)


Rows: 1137, unique pitches: 43
Parsed DataFrames:
- 00_schubert_d911_20: rows=1137, unique_pitches=43, min_duration=0.125


In [56]:
list(dfs_by_name)


['00_schubert_d911_20']

In [57]:

list(FILE_SOURCES)

['https://raw.githubusercontent.com/piasteuck/winterreise-analysis/refs/heads/main/Schubert_Winterreise_Dataset_v2-0/01_RawData/score_musicxml/Schubert_D911-20.xml']

In [58]:
from py_scripts import (
    vrv_set_options, vrv_load_from_url, vrv_render_all_pages, vrv_display_svg
)

url = FILE_SOURCES[1]

vrv_set_options(pageWidth=12000, pageHeight=1200, scale=40, breaks="none")
_ = vrv_load_from_url(url)  # auto detects 'mei' | 'musicxml' | 'humdrum'
for svg in vrv_render_all_pages():
    vrv_display_svg(svg)

IndexError: list index out of range

In [ ]:
df1 = results[0]['df']
df2 = results[1]['df']

display(df1)
display(df2)

,Measure,Local Onset,Global Onset,Duration,Pitch,Pitch Enharmonic,MIDI,Voice
0,1,0.0,0.0,8.0,G3,G3,55,Voice - Voice 1
1,1,8.0,8.0,4.0,F3,F3,53,Voice - Voice 1
2,1,12.0,12.0,3.0,G3,G3,55,Voice - Voice 1
3,1,12.0,12.0,8.0,G4,G4,67,Voice - Voice 1
4,1,15.0,15.0,1.0,A3,A3,57,Voice - Voice 1
...,...,...,...,...,...,...,...,...
750,1,666.0,666.0,1.0,F#4,G4,66,Voice - Voice 1
751,1,667.0,667.0,1.0,E4,E4,64,Voice - Voice 1
752,1,668.0,668.0,12.0,G2,D3,43,Voice - Voice 1
753,1,668.0,668.0,12.0,G3,Bb3,55,Voice - Voice 1


,Measure,Local Onset,Global Onset,Duration,Pitch,Pitch Enharmonic,MIDI,Voice,xml_id
0,1,3.0,-1.0,0.5,D4,F4,62,P2 - Voice 2,d1e93
1,1,3.0,-1.0,1.0,F4,D4,65,P2 - Voice 1,d1e92
2,1,3.0,-1.0,1.0,A4,D5,69,P1 - Voice 2,d1e91
3,1,3.0,-1.0,1.0,D5,A4,74,P1 - Voice 1,d1e64
4,1,3.5,-0.5,0.5,C4,C4,60,P2 - Voice 2,d1e94
...,...,...,...,...,...,...,...,...,...
230,4,37.5,45.5,0.5,G3,G3,55,P2 - Voice 1,d1e4-2
231,4,38.0,46.0,1.0,D2,F3,38,P2 - Voice 2,d1e6
232,4,38.0,46.0,1.0,F3,D2,53,P2 - Voice 1,d1e4-1
233,4,38.0,46.0,1.0,A3,C4,57,P1 - Voice 2,d1e4148


In [ ]:
df2.loc[df2['Pitch Enharmonic'].eq('Cb5')]

,Measure,Local Onset,Global Onset,Duration,Pitch,Pitch Enharmonic,MIDI,Voice,xml_id


In [ ]:
# Showcase: filtering/selecting from a notes DataFrame + piano-roll plot
# - by Global Onset range
# - by Measure range
# - by Voice(s)
# - plot a piano roll of the selection using py_scripts.music_utils

import pandas as pd
from IPython.display import display
from py_scripts.music_utils import draw_piano_roll

# Pick which parsed DataFrame to work with (e.g., df1 or df2 from above)
source_df = df2  # change to df1 to use the other score

# 1) Select by Global Onset range [lo, hi)
global_onset_lo, global_onset_hi = 0.0, 32.0
selected_by_onset = source_df[(source_df['Global Onset'] >= global_onset_lo) & (source_df['Global Onset'] < global_onset_hi)].copy()
print(f"Rows in Global Onset [{global_onset_lo}, {global_onset_hi}): {len(selected_by_onset)}")
display(selected_by_onset.head())

# 2) Select by Measure range (inclusive)
measure_lo, measure_hi = 1, 4
selected_by_measure = source_df[(source_df['Measure'] >= measure_lo) & (source_df['Measure'] <= measure_hi)].copy()
print(f"Rows in Measure [{measure_lo}, {measure_hi}]: {len(selected_by_measure)}")
display(selected_by_measure.head())

# 3) Voice exploration + filtering
voices_available = sorted(source_df['Voice'].dropna().unique().tolist())
print("Unique Voice types:")
display(pd.Series(voices_available, name='Voice types'))

# Filter by a specific voice label pattern (example: Voice 2)
voice_regex = r"Voice 2"  # matches 'Voice 2' within labels like 'P0 - Voice 2'
voice_filtered = source_df[source_df['Voice'].str.contains(voice_regex, regex=True, na=False)].copy()
print(f"Rows matching voices ~ /{voice_regex}/: {len(voice_filtered)}")
display(voice_filtered.head())

# Combine: onset window AND voice filter
selection = selected_by_onset[selected_by_onset['Voice'].str.contains(voice_regex, regex=True, na=False)].copy()
if selection.empty:
    # Fallback: if no notes match the voice in this time window, just show the time window
    selection = selected_by_onset
print(f"Final selection rows (combined filters where possible): {len(selection)}")

# Measure guide lines for selected window
# Prefer parser-provided measure_offsets if available for this source
try:
    res_idx = next((i for i, item in enumerate(results) if item.get('df') is source_df), None)
except Exception:
    res_idx = None

if res_idx is not None and isinstance(res_idx, int) and res_idx >= 0 and res_idx < len(results) and 'measure_offsets' in results[res_idx]:
    measure_offsets_full = list(results[res_idx]['measure_offsets'])
else:
    measure_offsets_full = (
        source_df.groupby('Measure')['Global Onset'].min().sort_values().tolist()
    )

# Compute extended right bound to include full duration of last note in selection
try:
    selection_end = float((selection['Global Onset'] + selection['Duration']).max())
except Exception:
    selection_end = None
extended_hi = max(global_onset_hi, selection_end) if selection_end is not None else global_onset_hi

# Restrict to the visible window [lo, extended_hi], inclusive of the right boundary
eps = 1e-6
visible_measure_offsets = [m for m in measure_offsets_full if (m >= global_onset_lo - eps) and (m <= extended_hi + eps)]
# Deduplicate with rounding for stability
if visible_measure_offsets:
    visible_measure_offsets = sorted({round(float(m), 6) for m in visible_measure_offsets})

# 4) Piano roll for the selection (uses existing utility; backend honors global PLOTTING_BACKEND)
backend = (PLOTTING_BACKEND.lower() if 'PLOTTING_BACKEND' in globals() else 'plt')
draw_piano_roll(
    selection,
    measure_offsets=visible_measure_offsets,
    backend=backend,
    plot_width=(PLOT_SIZE_X if 'PLOT_SIZE_X' in globals() else None),
    plot_height=(PLOT_SIZE_Y if 'PLOT_SIZE_Y' in globals() else None),
    zoom_drag_dim=ZOOM_DRAG_DIM,
    zoom_wheel_dim=ZOOM_WHEEL_DIM,
    show_measure_lines=True,
)

Rows in Global Onset [0.0, 32.0): 157


,Measure,Local Onset,Global Onset,Duration,Pitch,Pitch Enharmonic,MIDI,Voice,xml_id
5,1,0.0,0.0,1.0,B3,F4,59,P2 - Voice 2,d1e59
6,1,0.0,0.0,1.0,D4,D5,62,P1 - Voice 2,d1e487
7,1,0.0,0.0,1.0,F4,B3,65,P2 - Voice 1,d1e54
8,1,0.0,0.0,1.0,D5,D4,74,P1 - Voice 1,d1e366
9,1,1.0,1.0,0.5,A3,B3,57,P2 - Voice 2,d1e60


Rows in Measure [1, 4]: 235


,Measure,Local Onset,Global Onset,Duration,Pitch,Pitch Enharmonic,MIDI,Voice,xml_id
0,1,3.0,-1.0,0.5,D4,F4,62,P2 - Voice 2,d1e93
1,1,3.0,-1.0,1.0,F4,D4,65,P2 - Voice 1,d1e92
2,1,3.0,-1.0,1.0,A4,D5,69,P1 - Voice 2,d1e91
3,1,3.0,-1.0,1.0,D5,A4,74,P1 - Voice 1,d1e64
4,1,3.5,-0.5,0.5,C4,C4,60,P2 - Voice 2,d1e94


Unique Voice types:


0    P1 - Voice 1
1    P1 - Voice 2
2    P2 - Voice 1
3    P2 - Voice 2
Name: Voice types, dtype: object

Rows matching voices ~ /Voice 2/: 120


,Measure,Local Onset,Global Onset,Duration,Pitch,Pitch Enharmonic,MIDI,Voice,xml_id
0,1,3.0,-1.0,0.5,D4,F4,62,P2 - Voice 2,d1e93
2,1,3.0,-1.0,1.0,A4,D5,69,P1 - Voice 2,d1e91
4,1,3.5,-0.5,0.5,C4,C4,60,P2 - Voice 2,d1e94
5,1,0.0,0.0,1.0,B3,F4,59,P2 - Voice 2,d1e59
6,1,0.0,0.0,1.0,D4,D5,62,P1 - Voice 2,d1e487


Final selection rows (combined filters where possible): 81


In [ ]:
# Refactored pitch distribution using py_scripts.analysis_utils
from py_scripts import display_pitch_distribution

# Configuration for this cell (all lowercase options)
#
# PITCH_X_AXIS controls which pitch representation is used on the X axis:
#   - 'midi': use numeric MIDI values. If 'MIDI' column is missing, it is derived from names.
#   - 'pitch real' (synonyms: 'pitch', 'real'): use real pitch names corresponding to actual semitone values;
#       if only MIDI is present, names are derived from MIDI. Octave is included in the label.
#   - 'pitch enharmonic' (synonyms: 'pitch by name', 'pitch name', 'name', 'pitch_enharmonic', 'pitch-enharmonic'):
#       use written/enharmonic pitch names as notated (prefers 'Pitch Enharmonic' column),
#       preserving the spelled accidentals and octave, even if enharmonically different from real pitch.
PITCH_X_AXIS = 'pitch enharmonic'
#
# ORDER_X_AXIS_BY controls sort order of the X categories:
#   - 'midi' (synonyms: 'pitch real', 'pitch', 'real'):
#       sort by real MIDI number ascending (identical for all synonyms).
#   - 'pitch by octave real' (synonyms: 'pitch by octave', 'pitch octave', 'octave pitch', 'octave real'):
#       sort by REAL octave (computed from MIDI), then by pitch class within each octave.
#       This cares about actual pitch height. Example: B##3 resolves to its real octave (4); Cbb4 resolves to 3.
#   - 'pitch by octave enharmonic' (synonyms: 'octave enharmonic', 'octave name'):
#       sort by WRITTEN octave parsed from the name (ignoring semitone shifts of accidentals),
#       then by letter and accidental. Example: B##3 stays in octave 3; Cbb4 stays in octave 4.
#   - 'pitch by name' (synonyms: 'pitch enharmonic', 'enharmonic', 'enharmonic pitch', 'pitch_enharmonic'):
#       sort by letter (C..B) and accidental rank (bbb < bb < b < natural < # < ## < ###),
#       ignoring octave; ties are stabilized by the full label.
ORDER_X_AXIS_BY = 'pitch by name'

# Plotting backend
PLOTTING_BACKEND = 'bokeh'        # 'plt' | 'bokeh' | 'none'

# Plot sizing for this cell (universal for both backends)
PLOT_SIZE_X = 1200
PLOT_SIZE_Y = 600

# Bokeh hover config
BOKEH_HOVER_VALUES = True

# Choose the DataFrame to visualize
source = selection  # e.g., df1, results[0]['df'], dfs_by_name['00_some_name'] or a selection from df

counts_df = display_pitch_distribution(
    source,
    pitch_axis=PITCH_X_AXIS,
    order_axis_by=ORDER_X_AXIS_BY,
    backend=(PLOTTING_BACKEND.lower() if 'PLOTTING_BACKEND' in globals() else 'plt'),
    plot_width=PLOT_SIZE_X,
    plot_height=PLOT_SIZE_Y,
    show_hover=('BOKEH_HOVER_VALUES' in globals() and BOKEH_HOVER_VALUES),
    show_table=False,
)

Loading BokehJS ...

In [ ]:
# Duration distribution: count and visualize as bar plot (plt or bokeh)
# Uses global PLOTTING_BACKEND ('plt' or 'bokeh')

import numpy as np
import pandas as pd

# Configuration for this cell
DROP_ZERO_DURATIONS = True          # drop duration == 0
DURATION_ROUND_DECIMALS = 4         # round durations for cleaner categories; set None to disable

# Ensure df1 exists
if 'df1' not in globals():
    df1 = results[2]['df']

# Detect duration column
duration_col = None
for candidate in ['Duration', 'duration', 'durations', 'Durations']:
    if candidate in df1.columns:
        duration_col = candidate
        break

if duration_col is None:
    print("No duration column found. Checked: 'Duration', 'duration', 'durations', 'Durations'.")
else:
    ser = df1[duration_col].copy()
    # Coerce to numeric where possible
    ser = pd.to_numeric(ser, errors='coerce')
    ser = ser.dropna()
    if DROP_ZERO_DURATIONS:
        ser = ser[ser != 0]
    if DURATION_ROUND_DECIMALS is not None:
        ser = ser.round(DURATION_ROUND_DECIMALS)

    # Aggregate and sort by numeric value (ascending)
    counts = ser.value_counts().sort_index()

    # Display table
    display(counts.rename('count').to_frame())

    backend = (PLOTTING_BACKEND.lower() if 'PLOTTING_BACKEND' in globals() else 'plt')

    x_vals = counts.index.astype(str).tolist()
    y_vals = counts.values.tolist()

    if backend == 'plt':
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.bar(x_vals, y_vals, color='seagreen')
        ax.set_xlabel(duration_col)
        ax.set_ylabel('Count')
        ax.set_title('Duration Distribution')
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.show()

    elif backend == 'bokeh':
        from bokeh.plotting import figure, show
        from bokeh.io import output_notebook
        from bokeh.models import ColumnDataSource
        output_notebook()

        source = ColumnDataSource(dict(x=x_vals, count=y_vals))
        p = figure(x_range=x_vals, height=350, width=900, title='Duration Distribution', toolbar_location='right')
        p.vbar(x='x', top='count', width=0.9, source=source, fill_color='#2E8B57')
        p.xaxis.axis_label = duration_col
        p.yaxis.axis_label = 'Count'
        p.xgrid.grid_line_color = None
        p.y_range.start = 0
        show(p)

    else:
        print(f"Unsupported plotting backend: {backend}. Use 'plt' or 'bokeh'.")



,count
Duration,
0.5,8
1.0,163
2.0,270
3.0,55
4.0,186
6.0,12
8.0,44
12.0,11
16.0,4


Loading BokehJS ...

to do: enharmonic, intervall plot, combined plots